<a href="https://colab.research.google.com/github/naheejeong/cs182-final-project/blob/main/qwen35_9b_00_setup_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qwen3.5-9B, step 00: setup check

Same purpose as `00_llama_exploration.ipynb`, for `Qwen/Qwen3.5-9B`: check that we can download the model, that it fits on the L4, and that we understand its chat format before plugging it into InjecAgent.

Three things are different from Llama, and this notebook checks each one:

1. **Access.** Qwen3.5-9B is Apache-2.0 and not gated, so there is no access request to wait for.
2. **Loading.** It is a text + vision model. Its model card loads it with `AutoModelForMultimodalLM`, not `AutoModelForCausalLM`.
3. **Thinking mode.** It writes `<think>...</think>` before answering unless we pass `enable_thinking=False` to the chat template.

In [1]:
# hardware check - GPU & memory assigned
!nvidia-smi

Sun Nov 22 23:36:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   44C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Connect Google Drive to the Colab machine so results are saved to Drive.
from google.colab import drive
drive.mount('/content/drive')

# Setup commands (! = system command, % = notebook command)
%cd /content

# Download the InjecAgent benchmark (attack cases and evaluation code) from GitHub.
# --> this makes a new folder named "InjecAgent" in /content.
!git clone https://github.com/uiuc-kang-lab/InjecAgent.git
%cd /content/InjecAgent

# Install the 6 required packages (4 from requirements.txt + together and nltk).
!pip install -q -r requirements.txt together nltk

# Create a results folder in Google Drive (My Drive/cs182/results).
!mkdir -p /content/drive/MyDrive/cs182/results

# Create shortcut: InjecAgent/results -> My Drive/cs182/results, so outputs are saved to Drive.
# Qwen's results go into their own sub-folder (named after the model), so the Llama results are not touched.
!ln -sfn /content/drive/MyDrive/cs182/results results

# Provide environment variables needed by the InjecAgent code.
import os
os.environ["PYTHONPATH"] = "/content/InjecAgent"
os.environ["OPENAI_API_KEY"] = "unused" # placeholder; we never use!

!ls

Mounted at /content/drive
/content
Cloning into 'InjecAgent'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 102 (delta 46), reused 35 (delta 35), pack-reused 20 (from 1)
Receiving objects: 100% (102/102), 1.56 MiB | 20.73 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/InjecAgent
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
asset  data  LICENCE  README.md  requirements.txt  results  src


In [3]:
# Log in to HF using the token stored in Secrets. Token name: HF_TOKEN, username: naheejeong
# Qwen3.5-9B is Apache-2.0 and not gated, so no access request is needed (unlike Llama).
# We still log in, because logged-in downloads are less likely to be rate-limited.
from google.colab import userdata
from huggingface_hub import login, whoami, hf_hub_download
login(token=userdata.get('HF_TOKEN'))
print("Logged in as:", whoami()['name'])

# Testing Qwen access - download one tiny file from the model's page.
model_name = "Qwen/Qwen3.5-9B"
config_path = hf_hub_download(model_name, "config.json")
print(config_path)
print(open(config_path).read())    # printing the config file

Logged in as: naheejeong


config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

/root/.cache/huggingface/hub/models--Qwen--Qwen3.5-9B/snapshots/c202236235762e1c871ad0ccb60c8ee5ba337b9a/config.json
{
    "architectures": [
        "Qwen3_5ForConditionalGeneration"
    ],
    "image_token_id": 248056,
    "model_type": "qwen3_5",
    "text_config": {
        "attention_bias": false,
        "attention_dropout": 0.0,
        "attn_output_gate": true,
        "dtype": "bfloat16",
        "eos_token_id": 248044,
        "full_attention_interval": 4,
        "head_dim": 256,
        "hidden_act": "silu",
        "hidden_size": 4096,
        "initializer_range": 0.02,
        "intermediate_size": 12288,
        "layer_types": [
            "linear_attention",
            "linear_attention",
            "linear_attention",
            "full_attention",
            "linear_attention",
            "linear_attention",
            "linear_attention",
            "full_attention",
            "linear_attention",
            "linear_attention",
            "linear_attention",
 

In [4]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(model_name)   # downloads Qwen's tokenizer

ids = tok("This is a test input Meow :3")["input_ids"]   # turning text into ids
print(ids)
print(tok.convert_ids_to_tokens(ids))
print(tok.decode(ids))

# Tokens we rely on later.
print("eos token:", tok.eos_token, tok.eos_token_id)
print("pad token:", tok.pad_token, tok.pad_token_id)

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

[1919, 369, 264, 1228, 1879, 2088, 358, 535, 18]
['This', 'Ġis', 'Ġa', 'Ġtest', 'Ġinput', 'ĠMe', 'ow', 'Ġ:', '3']
This is a test input Meow :3
eos token: <|im_end|> 248046
pad token: <|endoftext|> 248044


In [5]:
# Chat format check, with thinking ON (Qwen's default) and OFF (what we will use).
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"},
]

prompt_on  = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False, enable_thinking=True)
prompt_off = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False, enable_thinking=False)

print("----- thinking ON -----")
print(prompt_on)
print("----- thinking OFF -----")
print(prompt_off)

# Check 1: the system message must appear in the prompt (InjecAgent puts its agent instructions there).
assert "You are a helpful assistant." in prompt_off, "system message is missing from the prompt"
# Check 2: the switch must actually change the prompt. If the two are identical, enable_thinking is being ignored.
assert prompt_on != prompt_off, "enable_thinking did not change the prompt"
print("both checks passed")

----- thinking ON -----
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
<think>

----- thinking OFF -----
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
<think>

</think>


both checks passed


In [6]:
from torch import bfloat16
from transformers import AutoModelForMultimodalLM
from transformers import __version__ as transformers_version

print("transformers version:", transformers_version)

# Download Qwen's weights (~19 GB) and load them onto the GPU at half precision.
# This is larger than Llama (~16 GB), so it is a tighter fit on the L4's 23 GB.
# If this cell fails with an "unknown architecture" style error, run:  !pip install -U transformers accelerate
# then restart the session and run the notebook again.
model = AutoModelForMultimodalLM.from_pretrained(
    model_name,
    dtype=bfloat16,
    device_map="auto",    # put the model on GPU automatically
)

!nvidia-smi

transformers version: 5.22.1


model.safetensors.index.json:   0%|          | 0.00/79.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

Sun Nov 22 23:38:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   45C    P0             28W /   72W |   18186MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
# Every part of the model should be on the GPU. Should print {'cuda'}.
# If it also prints 'cpu', the model did not fit and part of it was left on the CPU (very slow).
print({p.device.type for p in model.parameters()})

{'cuda'}


In [8]:
from torch import no_grad

def ask(messages, enable_thinking, max_new_tokens=512):
    # 1. Format the messages AND turn them into token IDs
    inputs = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    # 2. Let Qwen generate the continuation
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    with no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=pad_id)
    # 3. Keep only the new tokens (the reply) and turn them back into text
    prompt_length = inputs["input_ids"].shape[-1]
    print("prompt length in tokens:", prompt_length, "| reply length in tokens:", out.shape[-1] - prompt_length)
    return tok.decode(out[0][prompt_length:], skip_special_tokens=True)

# Thinking OFF: this is the setting we use for InjecAgent.
reply = ask(messages, enable_thinking=False)
print(reply)
assert "<think>" not in reply and "</think>" not in reply, "thinking text leaked into the reply"

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


prompt length in tokens: 30 | reply length in tokens: 69
The capital of France is **Paris**.

It is the country's most populous city and serves as its center for politics, economy, culture, and science. Located in the north-central part of the country along the Seine River, Paris is also a major global city known for its art, fashion, gastronomy, and history.



In [9]:
# Thinking ON, only to see what it looks like. The reply starts with the model's reasoning.
# InjecAgent's parser expects the reply to start with "Thought:" / "Action:", so this would be labelled invalid.
print(ask(messages, enable_thinking=True))

prompt length in tokens: 28 | reply length in tokens: 122
Thinking Process:

1.  **Identify the core question:** The user is asking "What is the capital of France?"
2.  **Access knowledge base:** Retrieve information about France's capital.
3.  **Verify information:** The capital of France is Paris.
4.  **Formulate the answer:** State the answer clearly and concisely.
5.  **Final Output:** "The capital of France is Paris." or simply "Paris."

Decision: Provide a direct and polite answer.cw
</think>

The capital of France is **Paris**.



## Notes

* Qwen's model card recommends sampling (temperature 0.7, top-p 0.8 in non-thinking mode). We use greedy decoding (`do_sample=False`) instead, to match the InjecAgent paper and our Llama run and to make results repeatable. Transformers may print a warning that the sampling settings are ignored; that is expected.
* Record here: GPU memory used after loading, and whether both checks in the chat-format cell passed.